# ANCIENT5_BUILDER

This notebook builds **ANCIENT5** as the definitive Ancient dataset expansion.

It stays coherent with the original `ancient3` method:

```text
DragonLLM/Clean-Wikipedia-English-Articles
+ Wikidata QID selection
+ filter by QID
= ANCIENT5
```

ANCIENT5 starts from `Datasets/ancient3_v4` and expands it with a controlled target of roughly 2k-3k new documents, using:

```text
1. domain-driven Wikidata discovery
2. question-driven QIDs from logs/ancient_TO_ADD.csv and related files
```

Recommended first pass: run through candidate selection, inspect `logs/ANCIENT5_selected_qids.csv`, then set `RUN_FETCH_AND_BUILD_DATASET = True`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 0. Setup

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import json
import math
import os
import re
import time

import pandas as pd
import requests
from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset, load_from_disk

PROJECT_ROOT_OVERRIDE = None
PROJECT_ROOT_CANDIDATES = [
    Path.cwd(),
    Path('/content/drive/MyDrive/NLP'),
    Path('/content/drive/MyDrive/Colab Notebooks/NLP'),
    Path('/gdrive/MyDrive/NLP'),
    Path('/gdrive/MyDrive/Colab Notebooks/NLP'),
]
PROJECT_MARKER = Path('Datasets') / 'ancient3_v4'

if PROJECT_ROOT_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser()
else:
    PROJECT_ROOT = next((p for p in PROJECT_ROOT_CANDIDATES if (p / PROJECT_MARKER).exists()), Path.cwd())

if not (PROJECT_ROOT / PROJECT_MARKER).exists():
    checked = '\n'.join(str(p / PROJECT_MARKER) for p in PROJECT_ROOT_CANDIDATES)
    raise FileNotFoundError(
        'Could not find Datasets/ancient3_v4.\n'
        f'Current working directory: {Path.cwd()}\n'
        'Mount Google Drive and rerun this cell, or set PROJECT_ROOT_OVERRIDE manually.\n'
        f'Checked:\n{checked}'
    )

os.chdir(PROJECT_ROOT)

LOGS_DIR = PROJECT_ROOT / 'logs'
DATASETS_DIR = PROJECT_ROOT / 'Datasets'
INDEXES_DIR = PROJECT_ROOT / 'Indexes'
CACHE_DIR = LOGS_DIR / '.cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

RUN_LABEL = 'ANCIENT5'
BASE_DATASET_DIR = DATASETS_DIR / 'ancient3_v4'
OUTPUT_DATASET_DIR = DATASETS_DIR / RUN_LABEL
SOURCE_WIKI_DATASET = 'DragonLLM/Clean-Wikipedia-English-Articles'

RUN_WIKIDATA_DISCOVERY = True
RUN_FETCH_AND_BUILD_DATASET = True  # first inspect selected QIDs, then turn True
OVERWRITE_OUTPUT_DATASET = False

SELECT_MIN_NEW_QIDS = 2000
SELECT_MAX_NEW_QIDS = 3200
MIN_SITELINKS_DEFAULT = 5

MAX_SOURCE_ROWS_TO_SCAN = None
SOURCE_SCAN_PROGRESS_EVERY = 250_000
USER_AGENT = 'ANCIENT5-builder/1.0 (student NLP project; Wikidata + DragonLLM QID filtering)'

WIKIDATA_RAW_CSV = LOGS_DIR / 'ANCIENT5_wikidata_candidate_rows.csv'
CANDIDATE_QIDS_CSV = LOGS_DIR / 'ANCIENT5_candidate_qids.csv'
SELECTED_QIDS_CSV = LOGS_DIR / 'ANCIENT5_selected_qids.csv'
ADDED_ARTICLES_CSV = LOGS_DIR / 'ANCIENT5_added_articles.csv'
SKIPPED_ARTICLES_CSV = LOGS_DIR / 'ANCIENT5_skipped_articles.csv'
BUILD_REPORT_JSON = LOGS_DIR / 'ANCIENT5_build_report.json'
MANUAL_SEED_QIDS_CSV = LOGS_DIR / 'ANCIENT5_manual_seed_qids.csv'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('BASE_DATASET_DIR:', BASE_DATASET_DIR)
print('OUTPUT_DATASET_DIR:', OUTPUT_DATASET_DIR)
print('RUN_WIKIDATA_DISCOVERY:', RUN_WIKIDATA_DISCOVERY)
print('RUN_FETCH_AND_BUILD_DATASET:', RUN_FETCH_AND_BUILD_DATASET)
print('SELECT_MAX_NEW_QIDS:', SELECT_MAX_NEW_QIDS)


## 1. Load Base Dataset

`ancient3_v4` is the seed. ANCIENT5 never discards it.

In [ ]:
base_ds = load_from_disk(str(BASE_DATASET_DIR))['train']
base_df = base_ds.to_pandas().fillna('')
QID_RE = re.compile(r'^Q\d+$')


def normalize_title(value):
    value = '' if value is None else str(value)
    value = value.replace('_', ' ').strip().lower()
    return re.sub(r'\s+', ' ', value)


def canonical_url(value):
    value = '' if value is None else str(value).strip()
    if not value:
        return ''
    value = value.split('#', 1)[0].replace('http://', 'https://')
    return value.rstrip('/')


def first_wikipedia_url(value):
    value = '' if value is None else str(value)
    for part in re.split(r'\s*\|\s*|\s*;\s*|\s*,\s*', value):
        part = part.strip()
        if 'wikipedia.org/wiki/' in part:
            return part
    return value.strip()


def clean_qid(value):
    if value is None:
        return ''
    if isinstance(value, float) and math.isnan(value):
        return ''
    value = str(value).strip()
    if value.startswith('http://www.wikidata.org/entity/') or value.startswith('https://www.wikidata.org/entity/'):
        value = value.rsplit('/', 1)[-1]
    match = re.search(r'Q\d+', value)
    return match.group(0) if match else ''


def extract_row_qid(row):
    if isinstance(row, dict):
        for key in ['qid', 'wikidata_qid', 'entity', 'wikidata_id']:
            qid = clean_qid(row.get(key, ''))
            if qid:
                return qid
        pageprops = row.get('pageprops') or {}
        if isinstance(pageprops, dict):
            qid = clean_qid(pageprops.get('wikibase_item', ''))
            if qid:
                return qid
    return ''

base_qids = set()
for _, row in base_df.iterrows():
    qid = ''
    for col in ['qid', 'entity', 'wikidata_qid']:
        if col in base_df.columns:
            qid = clean_qid(row.get(col, ''))
            if qid:
                break
    if qid:
        base_qids.add(qid)

base_titles = {normalize_title(t) for t in base_df.get('title', pd.Series([], dtype=str)).astype(str)}
base_urls = {canonical_url(u) for u in base_df.get('url', pd.Series([], dtype=str)).astype(str) if canonical_url(u)}

print(base_ds)
print('base rows:', base_ds.num_rows)
print('base qids:', len(base_qids))
print('base titles:', len(base_titles))
print('columns:', base_ds.column_names)
cols = [c for c in ['title', 'qid', 'entity', 'civilization', 'entity_type', 'topic'] if c in base_df.columns]
display(base_df[cols].head(10) if cols else base_df.head(10))

## 2. Wikidata Query Infrastructure

The cache avoids repeating Wikidata calls while tuning thresholds and quotas.

In [ ]:
WDQS_URL = 'https://query.wikidata.org/sparql'
WDQS_CACHE_JSON = CACHE_DIR / 'ANCIENT5_wdqs_cache.json'
WDQS_CACHE = json.loads(WDQS_CACHE_JSON.read_text(encoding='utf-8')) if WDQS_CACHE_JSON.exists() else {}


def save_wdqs_cache():
    WDQS_CACHE_JSON.write_text(json.dumps(WDQS_CACHE, indent=2, ensure_ascii=False), encoding='utf-8')


def make_sparql(where_block, limit=1000, min_sitelinks=MIN_SITELINKS_DEFAULT):
    return f'''
SELECT DISTINCT ?item ?itemLabel ?article ?sitelinks WHERE {{
  {where_block}

  ?article schema:about ?item ;
           schema:isPartOf <https://en.wikipedia.org/> .

  ?item wikibase:sitelinks ?sitelinks .
  FILTER(?sitelinks >= {int(min_sitelinks)})

  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
ORDER BY DESC(?sitelinks)
LIMIT {int(limit)}
'''


def run_wdqs_query(query_name, macro_area, query, retries=6, sleep_s=8):
    cache_key = query_name + '::' + str(abs(hash(query)))
    if cache_key in WDQS_CACHE:
        rows = WDQS_CACHE[cache_key]
        print(f'cache {query_name}: {len(rows)} rows')
        return rows

    headers = {'User-Agent': USER_AGENT, 'Accept': 'application/sparql-results+json'}
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(WDQS_URL, headers=headers, params={'query': query, 'format': 'json'}, timeout=90)
            if response.status_code == 429:
                wait = sleep_s * attempt
                print(f'WDQS 429 for {query_name}. Waiting {wait}s before retry {attempt}/{retries}...')
                time.sleep(wait)
                continue
            response.raise_for_status()
            data = response.json()
            out = []
            for b in data.get('results', {}).get('bindings', []):
                qid = clean_qid(b.get('item', {}).get('value', ''))
                if not qid:
                    continue
                out.append({
                    'qid': qid,
                    'label': b.get('itemLabel', {}).get('value', ''),
                    'wikipedia_url': b.get('article', {}).get('value', ''),
                    'sitelinks': int(float(b.get('sitelinks', {}).get('value', 0) or 0)),
                    'macro_area': macro_area,
                    'source_query': query_name,
                    'source_channel': 'wikidata_broad',
                })
            WDQS_CACHE[cache_key] = out
            save_wdqs_cache()
            print(f'fetched {query_name}: {len(out)} rows')
            time.sleep(1.0)
            return out
        except Exception as e:
            wait = sleep_s * attempt
            print(f'WDQS error for {query_name}: {type(e).__name__}: {e}. Waiting {wait}s...')
            time.sleep(wait)
    raise RuntimeError(f'Failed WDQS query after retries: {query_name}')


def wd_values(qids):
    return ' '.join(f'wd:{q}' for q in qids if clean_qid(q))

print('WDQS cache entries:', len(WDQS_CACHE))

## 3. Discovery Specs

This expands the original QID generation step across the semantic areas we care about.

In [ ]:
ANCHOR_QIDS = {
    'roman_world': ['Q1747689', 'Q2277', 'Q17167', 'Q1364601'],
    'ancient_greece': ['Q11772', 'Q171061', 'Q134178', 'Q1524', 'Q5690'],
    'ancient_egypt': ['Q11768', 'Q191324', 'Q726500'],
    'mesopotamia_near_east': ['Q11767', 'Q35355', 'Q4461035', 'Q47690', 'Q41137', 'Q34009', 'Q389688', 'Q41642', 'Q10798', 'Q32047'],
}
ALL_ANCHORS = sorted(set(sum(ANCHOR_QIDS.values(), [])))

CLASS_QIDS = {
    'people_occupations': ['Q496418', 'Q482980', 'Q49757', 'Q36180', 'Q82955', 'Q47064', 'Q42603', 'Q37226', 'Q1281618'],
    'works': ['Q386724', 'Q7725634', 'Q571', 'Q5185279', 'Q37484', 'Q179461'],
    'events': ['Q178561', 'Q198', 'Q831663', 'Q188055', 'Q124734'],
    'religion_mythology': ['Q178885', 'Q4271324', 'Q9134', 'Q9174'],
    'places_architecture': ['Q515', 'Q486972', 'Q839954', 'Q811979', 'Q41176', 'Q44539', 'Q4989906'],
    'languages_scripts': ['Q34770', 'Q8192', 'Q9779', 'Q33384'],
    'law_society': ['Q7748', 'Q178706', 'Q28108', 'Q874405', 'Q16334295'],
    'technology_science': ['Q11016', 'Q205375', 'Q39546', 'Q728', 'Q488383', 'Q11019'],
}

LANGUAGE_SCRIPT_SEEDS = ['Q35497', 'Q397', 'Q8216', 'Q401', 'Q170342', 'Q35518', 'Q36790', 'Q28602']


def anchor_relation_block(anchor_qids):
    anchors = wd_values(anchor_qids)
    return f'''
  VALUES ?anchor {{ {anchors} }}
  {{ ?item wdt:P361+ ?anchor . }}
  UNION {{ ?item wdt:P17 ?anchor . }}
  UNION {{ ?item wdt:P131+ ?anchor . }}
  UNION {{ ?item wdt:P27 ?anchor . }}
  UNION {{ ?item wdt:P921 ?anchor . }}
'''


def class_anchor_query(class_qids, anchor_qids):
    classes = wd_values(class_qids)
    anchors = wd_values(anchor_qids)
    return f'''
  VALUES ?class {{ {classes} }}
  VALUES ?anchor {{ {anchors} }}
  ?item wdt:P31/wdt:P279* ?class .
  {{ ?item wdt:P361+ ?anchor . }}
  UNION {{ ?item wdt:P17 ?anchor . }}
  UNION {{ ?item wdt:P131+ ?anchor . }}
  UNION {{ ?item wdt:P276/wdt:P131* ?anchor . }}
  UNION {{ ?item wdt:P921 ?anchor . }}
  UNION {{ ?item wdt:P840 ?anchor . }}
'''


def people_by_civilization_query(anchor_qids):
    anchors = wd_values(anchor_qids)
    return f'''
  VALUES ?anchor {{ {anchors} }}
  ?item wdt:P31 wd:Q5 .
  {{ ?item wdt:P27 ?anchor . }}
  UNION {{ ?item wdt:P19/wdt:P131* ?anchor . }}
  UNION {{ ?item wdt:P937/wdt:P131* ?anchor . }}
'''


def people_by_occupation_and_time_query(anchor_qids, occupation_qids):
    anchors = wd_values(anchor_qids)
    occupations = wd_values(occupation_qids)
    return f'''
  VALUES ?anchor {{ {anchors} }}
  VALUES ?occupation {{ {occupations} }}
  ?item wdt:P31 wd:Q5 ;
        wdt:P106 ?occupation .
  OPTIONAL {{ ?item wdt:P569 ?birth . }}
  {{ ?item wdt:P27 ?anchor . }}
  UNION {{ ?item wdt:P19/wdt:P131* ?anchor . }}
  UNION {{ ?item wdt:P937/wdt:P131* ?anchor . }}
  FILTER(!BOUND(?birth) || YEAR(?birth) < 800)
'''


def works_by_author_or_anchor_query(anchor_qids, work_class_qids):
    anchors = wd_values(anchor_qids)
    classes = wd_values(work_class_qids)
    return f'''
  VALUES ?anchor {{ {anchors} }}
  VALUES ?class {{ {classes} }}
  ?item wdt:P31/wdt:P279* ?class .
  {{ ?item wdt:P921 ?anchor . }}
  UNION {{ ?item wdt:P361+ ?anchor . }}
  UNION {{ ?item wdt:P840 ?anchor . }}
  UNION {{ ?item wdt:P50 ?author . ?author wdt:P27 ?anchor . }}
  UNION {{ ?item wdt:P50 ?author . ?author wdt:P19/wdt:P131* ?anchor . }}
  UNION {{ ?item wdt:P50 ?author . ?author wdt:P569 ?birth . FILTER(YEAR(?birth) < 800) }}
'''


def language_seed_query(seed_qids):
    seeds = wd_values(seed_qids)
    return f'''
  VALUES ?seed {{ {seeds} }}
  {{ BIND(?seed AS ?item) }}
  UNION {{ ?item wdt:P361+ ?seed . }}
  UNION {{ ?item wdt:P279* ?seed . }}
  UNION {{ ?item wdt:P31/wdt:P279* ?seed . }}
'''

QUERY_SPECS = []
for macro_area, qids in ANCHOR_QIDS.items():
    limit = 1600 if macro_area != 'roman_world' else 1200
    QUERY_SPECS.append({'name': f'{macro_area}_core_relations', 'macro_area': macro_area, 'query': make_sparql(anchor_relation_block(qids), limit=limit)})

QUERY_SPECS.extend([
    {'name': 'people_by_civilization', 'macro_area': 'people', 'query': make_sparql(people_by_civilization_query(ALL_ANCHORS), limit=1400)},
    {'name': 'people_by_occupation_and_ancient_anchor', 'macro_area': 'people', 'query': make_sparql(people_by_occupation_and_time_query(ALL_ANCHORS, CLASS_QIDS['people_occupations']), limit=1400)},
    {'name': 'works_texts_by_author_or_anchor', 'macro_area': 'works_texts', 'query': make_sparql(works_by_author_or_anchor_query(ALL_ANCHORS, CLASS_QIDS['works']), limit=900)},
    {'name': 'events_warfare_by_anchor', 'macro_area': 'events_warfare', 'query': make_sparql(class_anchor_query(CLASS_QIDS['events'], ALL_ANCHORS), limit=900)},
    {'name': 'religion_mythology_by_anchor', 'macro_area': 'religion_mythology', 'query': make_sparql(class_anchor_query(CLASS_QIDS['religion_mythology'], ALL_ANCHORS), limit=700)},
    {'name': 'architecture_places_by_anchor', 'macro_area': 'architecture_places', 'query': make_sparql(class_anchor_query(CLASS_QIDS['places_architecture'], ALL_ANCHORS), limit=1200)},
    {'name': 'languages_scripts_seed_expansion', 'macro_area': 'languages_scripts', 'query': make_sparql(language_seed_query(LANGUAGE_SCRIPT_SEEDS), limit=500, min_sitelinks=2)},
    {'name': 'law_society_institutions_by_anchor', 'macro_area': 'law_society_institutions', 'query': make_sparql(class_anchor_query(CLASS_QIDS['law_society'], ALL_ANCHORS), limit=600)},
    {'name': 'technology_science_by_anchor', 'macro_area': 'technology_science', 'query': make_sparql(class_anchor_query(CLASS_QIDS['technology_science'], ALL_ANCHORS), limit=600)},
])

print('query specs:', len(QUERY_SPECS))
for spec in QUERY_SPECS:
    print('-', spec['name'], '=>', spec['macro_area'])

# ANCIENT5-lite discovery:
# Avoid large WDQS queries. Use small direct relation queries for undercovered civilizations.
# This is much more reliable than one huge UNION/transitive query.

HEAVY_QUERY_NAMES = {
    "roman_world_core_relations",
    "ancient_greece_core_relations",
    "ancient_egypt_core_relations",
    "mesopotamia_near_east_core_relations",
    "people_by_civilization",
    "people_by_occupation_and_ancient_anchor",
    "works_texts_by_author_or_anchor",
    "events_warfare_by_anchor",
    "religion_mythology_by_anchor",
    "architecture_places_by_anchor",
    "law_society_institutions_by_anchor",
    "technology_science_by_anchor",
}

# Keep only the language/script seed query from the original semantic specs.
QUERY_SPECS = [
    spec for spec in QUERY_SPECS
    if spec["name"] not in HEAVY_QUERY_NAMES
]

def make_sparql_light(where_block, limit=300, min_sitelinks=3):
    # No ORDER BY: much faster on Wikidata.
    return f"""
SELECT DISTINCT ?item ?itemLabel ?article ?sitelinks WHERE {{
  {where_block}

  ?article schema:about ?item ;
           schema:isPartOf <https://en.wikipedia.org/> .

  ?item wikibase:sitelinks ?sitelinks .
  FILTER(?sitelinks >= {int(min_sitelinks)})

  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
LIMIT {int(limit)}
"""

def small_relation_query(anchor_qids, relation_pattern):
    anchors = wd_values(anchor_qids)
    return f"""
  VALUES ?anchor {{ {anchors} }}
  {relation_pattern}
"""

SMALL_CORE_RELATIONS = [
    ("part_of", "?item wdt:P361 ?anchor ."),
    ("main_subject", "?item wdt:P921 ?anchor ."),
    ("country", "?item wdt:P17 ?anchor ."),
    ("located_in", "?item wdt:P131 ?anchor ."),
    ("citizenship", "?item wdt:P27 ?anchor ."),
]

SMALL_CORE_AREAS = {
    "ancient_greece": ANCHOR_QIDS["ancient_greece"],
    "ancient_egypt": ANCHOR_QIDS["ancient_egypt"],
    "mesopotamia_near_east": ANCHOR_QIDS["mesopotamia_near_east"],
}

small_specs = []
for area, anchors in SMALL_CORE_AREAS.items():
    for rel_name, rel_pattern in SMALL_CORE_RELATIONS:
        small_specs.append({
            "name": f"{area}_small_core_{rel_name}",
            "macro_area": area,
            "query": make_sparql_light(
                small_relation_query(anchors, rel_pattern),
                limit=300,
                min_sitelinks=3,
            ),
        })

# Put small reliable queries first, then language/script query.
QUERY_SPECS = small_specs + QUERY_SPECS

print("\nquery specs after ANCIENT5-lite rewrite:", len(QUERY_SPECS))
for spec in QUERY_SPECS:
    print("-", spec["name"], "=>", spec["macro_area"])


# Add semantic-lite queries: small, direct, no broad UNION, no ORDER BY.
# These are meant to add works/texts, people, events, religion, architecture,
# languages, law/society and technology without timing out WDQS.

def small_class_relation_query(class_qids, anchor_qids, relation_pattern):
    classes = wd_values(class_qids)
    anchors = wd_values(anchor_qids)
    return f"""
  VALUES ?class {{ {classes} }}
  VALUES ?anchor {{ {anchors} }}

  ?item wdt:P31/wdt:P279* ?class .
  {relation_pattern}
"""

def small_people_query(anchor_qids, occupation_qids):
    anchors = wd_values(anchor_qids)
    occupations = wd_values(occupation_qids)
    return f"""
  VALUES ?anchor {{ {anchors} }}
  VALUES ?occupation {{ {occupations} }}

  ?item wdt:P31 wd:Q5 ;
        wdt:P106 ?occupation ;
        wdt:P27 ?anchor .

  OPTIONAL {{ ?item wdt:P569 ?birth . }}
  FILTER(!BOUND(?birth) || YEAR(?birth) < 800)
"""

SEMANTIC_LITE_SPECS = []

SEMANTIC_AREAS = {
    "ancient_greece": ANCHOR_QIDS["ancient_greece"],
    "ancient_egypt": ANCHOR_QIDS["ancient_egypt"],
    "mesopotamia_near_east": ANCHOR_QIDS["mesopotamia_near_east"],
}

SEMANTIC_CLASSES = {
    "works_texts": CLASS_QIDS["works"],
    "events_warfare": CLASS_QIDS["events"],
    "religion_mythology": CLASS_QIDS["religion_mythology"],
    "architecture_places": CLASS_QIDS["places_architecture"],
    "law_society_institutions": CLASS_QIDS["law_society"],
    "technology_science": CLASS_QIDS["technology_science"],
}

# People: direct citizenship + occupation only. Much lighter than the previous people query.
for area, anchors in SEMANTIC_AREAS.items():
    SEMANTIC_LITE_SPECS.append({
        "name": f"{area}_semantic_people_occupation",
        "macro_area": "people",
        "query": make_sparql_light(
            small_people_query(anchors, CLASS_QIDS["people_occupations"]),
            limit=250,
            min_sitelinks=5,
        ),
    })

# Semantic class queries: direct part_of / main_subject only.
for semantic_area, class_qids in SEMANTIC_CLASSES.items():
    for civ_area, anchors in SEMANTIC_AREAS.items():
        for rel_name, rel_pattern in [
            ("part_of", "?item wdt:P361 ?anchor ."),
            ("main_subject", "?item wdt:P921 ?anchor ."),
        ]:
            SEMANTIC_LITE_SPECS.append({
                "name": f"{civ_area}_{semantic_area}_{rel_name}",
                "macro_area": semantic_area,
                "query": make_sparql_light(
                    small_class_relation_query(class_qids, anchors, rel_pattern),
                    limit=200,
                    min_sitelinks=3,
                ),
            })

# Languages/scripts: keep seed expansion, but add direct ancient subject links too.
for civ_area, anchors in SEMANTIC_AREAS.items():
    SEMANTIC_LITE_SPECS.append({
        "name": f"{civ_area}_languages_scripts_main_subject",
        "macro_area": "languages_scripts",
        "query": make_sparql_light(
            small_class_relation_query(
                CLASS_QIDS["languages_scripts"],
                anchors,
                "?item wdt:P921 ?anchor .",
            ),
            limit=200,
            min_sitelinks=2,
        ),
    })

QUERY_SPECS = QUERY_SPECS + SEMANTIC_LITE_SPECS

print("\nquery specs after adding semantic-lite:", len(QUERY_SPECS))
for spec in QUERY_SPECS:
    print("-", spec["name"], "=>", spec["macro_area"])

## 4. Run Wikidata Discovery

In [ ]:
# @title 4. Run Wikidata Discovery with progress bar

from tqdm.auto import tqdm

if RUN_WIKIDATA_DISCOVERY:
    raw_rows = []
    failed_queries = []

    progress = tqdm(QUERY_SPECS, desc="Wikidata discovery", unit="query")

    for spec in progress:
        progress.set_postfix({
            "current": spec["name"][:35],
            "found_rows": len(raw_rows),
            "failed": len(failed_queries),
        })

        try:
            rows = run_wdqs_query(
                spec["name"],
                spec["macro_area"],
                spec["query"],
            )
            raw_rows.extend(rows)

            progress.set_postfix({
                "last_ok": spec["name"][:35],
                "found_rows": len(raw_rows),
                "failed": len(failed_queries),
            })

        except RuntimeError as e:
            print(f"\nSKIPPED failed query: {spec['name']} | {e}")
            failed_queries.append({
                "name": spec["name"],
                "macro_area": spec["macro_area"],
                "error": str(e),
            })
            continue

    wikidata_raw = pd.DataFrame(raw_rows).drop_duplicates()

    if len(wikidata_raw):
        wikidata_raw["discovered_at_utc"] = datetime.now(timezone.utc).isoformat()
        wikidata_raw = wikidata_raw.sort_values(
            ["macro_area", "sitelinks"],
            ascending=[True, False],
        )

    wikidata_raw.to_csv(WIKIDATA_RAW_CSV, index=False)

    failed_queries_path = LOGS_DIR / "ANCIENT5_failed_wikidata_queries.json"
    failed_queries_path.write_text(
        json.dumps(failed_queries, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

else:
    if not WIKIDATA_RAW_CSV.exists():
        raise FileNotFoundError(
            f"{WIKIDATA_RAW_CSV} does not exist and RUN_WIKIDATA_DISCOVERY=False"
        )
    wikidata_raw = pd.read_csv(WIKIDATA_RAW_CSV).fillna("")
    failed_queries = []

print("wikidata raw rows:", len(wikidata_raw))
print("unique qids:", wikidata_raw["qid"].nunique() if len(wikidata_raw) else 0)
print("failed queries:", len(failed_queries))
print("saved:", WIKIDATA_RAW_CSV)

if len(wikidata_raw):
    display(wikidata_raw["macro_area"].value_counts().to_frame("raw_rows"))
    display(wikidata_raw.head(30))

if failed_queries:
    display(pd.DataFrame(failed_queries))

## 5. Add Question-Driven QIDs

In [ ]:
QUESTION_DRIVEN_COLUMNS = ['qid', 'label', 'wikipedia_url', 'sitelinks', 'macro_area', 'source_query', 'source_channel', 'question_ids', 'notes']
question_rows = []

to_add_path = LOGS_DIR / 'ancient_TO_ADD.csv'
if to_add_path.exists():
    to_add = pd.read_csv(to_add_path, on_bad_lines='skip').fillna('')
    for _, row in to_add.iterrows():
        qid = clean_qid(row.get('wikidata_qid', ''))
        if qid:
            question_rows.append({
                'qid': qid,
                'label': str(row.get('article_to_add', '') or row.get('resolved_title', '')).strip(),
                'wikipedia_url': first_wikipedia_url(row.get('wikipedia_url', '')),
                'sitelinks': 10_000,
                'macro_area': 'question_driven',
                'source_query': 'ancient_TO_ADD',
                'source_channel': 'question_driven',
                'question_ids': str(row.get('question_ids', '') or row.get('question_id', '')).strip(),
                'notes': str(row.get('notes', '')).strip(),
            })

missing_path = LOGS_DIR / 'ancient_missing_articles_to_add.csv'
if missing_path.exists():
    missing = pd.read_csv(missing_path).fillna('')
    for _, row in missing.iterrows():
        qid = clean_qid(row.get('wikidata_qid', ''))
        if qid:
            question_rows.append({
                'qid': qid,
                'label': str(row.get('article_to_add', '')).strip(),
                'wikipedia_url': first_wikipedia_url(row.get('wikipedia_url', '')),
                'sitelinks': 9_000,
                'macro_area': 'question_driven',
                'source_query': 'ancient_missing_articles_to_add',
                'source_channel': 'question_driven',
                'question_ids': str(row.get('question_ids', '')).strip(),
                'notes': 'from ancient_missing_articles_to_add.csv',
            })

if not MANUAL_SEED_QIDS_CSV.exists():
    pd.DataFrame(columns=['qid', 'label', 'wikipedia_url', 'macro_area', 'question_ids', 'notes']).to_csv(MANUAL_SEED_QIDS_CSV, index=False)
    print('created optional manual seed template:', MANUAL_SEED_QIDS_CSV)
else:
    manual = pd.read_csv(MANUAL_SEED_QIDS_CSV).fillna('')
    for _, row in manual.iterrows():
        qid = clean_qid(row.get('qid', '') or row.get('wikidata_qid', ''))
        if qid:
            question_rows.append({
                'qid': qid,
                'label': str(row.get('label', '') or row.get('article_to_add', '')).strip(),
                'wikipedia_url': first_wikipedia_url(row.get('wikipedia_url', '')),
                'sitelinks': 8_000,
                'macro_area': str(row.get('macro_area', '') or 'question_driven').strip(),
                'source_query': 'ANCIENT5_manual_seed_qids',
                'source_channel': 'question_driven_manual',
                'question_ids': str(row.get('question_ids', '')).strip(),
                'notes': str(row.get('notes', '')).strip(),
            })

question_driven = pd.DataFrame(question_rows, columns=QUESTION_DRIVEN_COLUMNS)
print('question-driven candidate rows:', len(question_driven))
if len(question_driven):
    display(question_driven.head(50))

## 6. Merge, Score, Select QIDs

Outputs:

```text
logs/ANCIENT5_candidate_qids.csv
logs/ANCIENT5_selected_qids.csv
```

In [ ]:
AREA_BUDGETS = {
    'question_driven': 800,
    'ancient_greece': 650,
    'ancient_egypt': 450,
    'mesopotamia_near_east': 450,
    'roman_world': 250,
    'people': 600,
    'works_texts': 450,
    'events_warfare': 350,
    'religion_mythology': 300,
    'architecture_places': 500,
    'languages_scripts': 250,
    'law_society_institutions': 250,
    'technology_science': 300,
}
AREA_PRIORITY = ['question_driven', 'ancient_greece', 'works_texts', 'languages_scripts', 'ancient_egypt', 'mesopotamia_near_east', 'people', 'events_warfare', 'religion_mythology', 'architecture_places', 'law_society_institutions', 'technology_science', 'roman_world']
AREA_BOOST = {'question_driven': 10_000, 'ancient_greece': 80, 'works_texts': 70, 'languages_scripts': 70, 'ancient_egypt': 55, 'mesopotamia_near_east': 55, 'people': 35, 'events_warfare': 35, 'religion_mythology': 35, 'architecture_places': 30, 'law_society_institutions': 25, 'technology_science': 25, 'roman_world': -25}


def split_pipe(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    return [x.strip() for x in re.split(r'\s*\|\s*', str(value)) if x.strip()]


def join_unique(values):
    out = []
    for value in values:
        for part in split_pipe(value):
            if part and part not in out:
                out.append(part)
    return ' | '.join(out)


def choose_primary_area(areas, channels):
    area_set = set(split_pipe(areas))
    channel_set = set(split_pipe(channels))
    if {'question_driven', 'question_driven_manual'} & channel_set or 'question_driven' in area_set:
        return 'question_driven'
    for area in AREA_PRIORITY:
        if area in area_set:
            return area
    return sorted(area_set)[0] if area_set else 'unknown'


def max_numeric(series):
    values = pd.to_numeric(series, errors='coerce')
    return int(values.max()) if values.notna().any() else 0

all_rows = []
if len(wikidata_raw):
    all_rows.append(wikidata_raw[['qid', 'label', 'wikipedia_url', 'sitelinks', 'macro_area', 'source_query', 'source_channel']].copy())
if len(question_driven):
    all_rows.append(question_driven[['qid', 'label', 'wikipedia_url', 'sitelinks', 'macro_area', 'source_query', 'source_channel', 'question_ids', 'notes']].copy())
if not all_rows:
    raise RuntimeError('No candidate rows. Run Wikidata discovery or provide question-driven/manual QIDs.')

candidate_rows = pd.concat(all_rows, ignore_index=True).fillna('')
candidate_rows['qid'] = candidate_rows['qid'].map(clean_qid)
candidate_rows = candidate_rows[candidate_rows['qid'].astype(bool)].copy()
for col in ['question_ids', 'notes']:
    if col not in candidate_rows.columns:
        candidate_rows[col] = ''

merged = []
for qid, group in candidate_rows.groupby('qid', dropna=False):
    labels = [x for x in group['label'].astype(str).tolist() if x and not x.startswith('Q')]
    urls = [first_wikipedia_url(x) for x in group['wikipedia_url'].astype(str).tolist() if first_wikipedia_url(x)]
    macro_areas = join_unique(group['macro_area'].tolist())
    source_queries = join_unique(group['source_query'].tolist())
    source_channels = join_unique(group['source_channel'].tolist())
    question_ids = join_unique(group.get('question_ids', pd.Series([], dtype=str)).tolist())
    notes = join_unique(group.get('notes', pd.Series([], dtype=str)).tolist())
    primary_area = choose_primary_area(macro_areas, source_channels)
    sitelinks = max_numeric(group['sitelinks'])
    source_count = group['source_query'].nunique()
    in_base = qid in base_qids
    title_norm = normalize_title(labels[0] if labels else '')
    url_norm = canonical_url(urls[0] if urls else '')
    if not in_base and title_norm and title_norm in base_titles:
        in_base = True
    if not in_base and url_norm and url_norm in base_urls:
        in_base = True
    score = min(sitelinks, 700) / 7.0 + source_count * 20 + AREA_BOOST.get(primary_area, 0)
    if in_base:
        score += 5000
    merged.append({
        'qid': qid,
        'label': labels[0] if labels else qid,
        'wikipedia_url': urls[0] if urls else '',
        'sitelinks': sitelinks,
        'macro_areas': macro_areas,
        'primary_macro_area': primary_area,
        'source_queries': source_queries,
        'source_channels': source_channels,
        'source_count': source_count,
        'question_ids': question_ids,
        'already_in_base': bool(in_base),
        'selection_score': round(score, 3),
        'notes': notes,
    })

candidates = pd.DataFrame(merged).sort_values('selection_score', ascending=False).reset_index(drop=True)
candidates['selected_for_ANCIENT5'] = False
candidates['selection_status'] = 'not_selected_quota'
candidates.loc[candidates['already_in_base'], 'selected_for_ANCIENT5'] = True
candidates.loc[candidates['already_in_base'], 'selection_status'] = 'base_seed'

new_pool = candidates[~candidates['already_in_base']].copy()
selected_indices = []
for area in AREA_PRIORITY:
    budget = AREA_BUDGETS.get(area, 0)
    area_pool = new_pool[(new_pool['primary_macro_area'] == area) & (~new_pool.index.isin(selected_indices))]
    take = area_pool.sort_values('selection_score', ascending=False).head(budget)
    selected_indices.extend(take.index.tolist())
    if len(selected_indices) >= SELECT_MAX_NEW_QIDS:
        selected_indices = selected_indices[:SELECT_MAX_NEW_QIDS]
        break
if len(selected_indices) < SELECT_MAX_NEW_QIDS:
    remaining = new_pool[~new_pool.index.isin(selected_indices)].sort_values('selection_score', ascending=False)
    selected_indices.extend(remaining.head(SELECT_MAX_NEW_QIDS - len(selected_indices)).index.tolist())

candidates.loc[selected_indices, 'selected_for_ANCIENT5'] = True
candidates.loc[selected_indices, 'selection_status'] = 'new_selected'
candidates['has_wikipedia_url'] = candidates['wikipedia_url'].astype(str).str.contains('wikipedia.org/wiki/', na=False)

selected = candidates[candidates['selected_for_ANCIENT5']].copy()
selected_new = candidates[candidates['selection_status'] == 'new_selected'].copy()

candidates.to_csv(CANDIDATE_QIDS_CSV, index=False)
selected.to_csv(SELECTED_QIDS_CSV, index=False)

print('candidate qids:', len(candidates))
print('base seed qids selected:', int((selected['selection_status'] == 'base_seed').sum()))
print('new qids selected:', len(selected_new))
print('saved candidates:', CANDIDATE_QIDS_CSV)
print('saved selected:', SELECTED_QIDS_CSV)
if len(selected_new) < SELECT_MIN_NEW_QIDS:
    print(f'WARNING: selected new QIDs below target: {len(selected_new)} < {SELECT_MIN_NEW_QIDS}')
print('\nSelected new QIDs by area:')
display(selected_new['primary_macro_area'].value_counts().to_frame('new_selected'))
display(candidates.head(50))

## 7. Audit Before Build

In [ ]:
selected = pd.read_csv(SELECTED_QIDS_CSV).fillna('')
selected_new = selected[selected['selection_status'] == 'new_selected'].copy()

print('ANCIENT5 audit')
print('selected total:', len(selected))
print('selected new:', len(selected_new))
print('already in base:', int((selected['selection_status'] == 'base_seed').sum()))
print('missing URL among selected new:', int((~selected_new['has_wikipedia_url'].astype(bool)).sum()) if 'has_wikipedia_url' in selected_new else 'n/a')
display(selected_new['primary_macro_area'].value_counts().to_frame('new_selected'))
display(selected_new[['qid', 'label', 'wikipedia_url', 'primary_macro_area', 'sitelinks', 'selection_score', 'source_queries']].head(100))
display(selected_new[selected_new['primary_macro_area'] == 'question_driven'][['qid', 'label', 'wikipedia_url', 'question_ids', 'source_queries', 'notes']].head(100))

if not RUN_FETCH_AND_BUILD_DATASET:
    print('\nSTOP HERE for first pass: inspect ANCIENT5_selected_qids.csv, then set RUN_FETCH_AND_BUILD_DATASET=True in setup.')

In [ ]:
selected = pd.read_csv(SELECTED_QIDS_CSV).fillna("")
selected_new = selected[selected["selection_status"] == "new_selected"].copy()

display(selected_new["primary_macro_area"].value_counts().to_frame("selected_new"))

display(
    selected_new[
        ["qid", "label", "wikipedia_url", "primary_macro_area", "sitelinks", "selection_score", "source_queries"]
    ].groupby("primary_macro_area").head(15)
)

## 8. Fetch Selected Articles From DragonLLM

If DragonLLM is gated, authenticate before running this section:

```python
from huggingface_hub import login
login()
```

In [ ]:
from huggingface_hub import login
login()

In [ ]:
selected = pd.read_csv(SELECTED_QIDS_CSV).fillna('')
selected_new = selected[selected['selection_status'] == 'new_selected'].copy()
target_qids = set(selected_new['qid'].map(clean_qid))
selected_by_qid = {row['qid']: row.to_dict() for _, row in selected_new.iterrows()}

candidate_rows = []
missing_after_fetch = selected_new.copy()

if not RUN_FETCH_AND_BUILD_DATASET:
    print('Skipping DragonLLM fetch because RUN_FETCH_AND_BUILD_DATASET=False')
else:
    print('target new qids:', len(target_qids))
    source_stream = load_dataset(SOURCE_WIKI_DATASET, split='train', streaming=True)
    found_by_qid = {}
    scanned = 0
    for row in source_stream:
        scanned += 1
        qid = extract_row_qid(row)
        if qid and qid in target_qids and qid not in found_by_qid:
            copied = dict(row)
            copied['source_mode'] = 'source_streaming'
            copied['source_expected_qid'] = qid
            found_by_qid[qid] = copied
            meta = selected_by_qid.get(qid, {})
            print(f'found {len(found_by_qid):>4}/{len(target_qids)}: {qid} | {meta.get("label", copied.get("title", ""))}')
            if len(found_by_qid) == len(target_qids):
                break
        if SOURCE_SCAN_PROGRESS_EVERY and scanned % SOURCE_SCAN_PROGRESS_EVERY == 0:
            print(f'scanned {scanned:,} rows; found {len(found_by_qid):,}/{len(target_qids):,}')
        if MAX_SOURCE_ROWS_TO_SCAN is not None and scanned >= MAX_SOURCE_ROWS_TO_SCAN:
            print(f'stopped after MAX_SOURCE_ROWS_TO_SCAN={MAX_SOURCE_ROWS_TO_SCAN}')
            break
    candidate_rows = list(found_by_qid.values())
    missing_after_fetch = selected_new[~selected_new['qid'].isin(set(found_by_qid))].copy()
    print('source rows scanned:', scanned)
    print('candidate rows fetched:', len(candidate_rows))
    print('selected qids still missing:', len(missing_after_fetch))
    if len(missing_after_fetch):
        display(missing_after_fetch[['qid', 'label', 'wikipedia_url', 'primary_macro_area', 'source_queries']].head(100))

In [ ]:
# @title Rescue the 14 manually approved QIDs from DragonLLM and append to candidate_rows
# DOPO cella 8, PRIMA cella 9.

import re
import pandas as pd
from datasets import load_dataset
from IPython.display import display

SOURCE_WIKI_SPLIT = "train"

FORCE_INCLUDE_QIDS = {
    "Q5555632",   # First Messenian War
    "Q34601",     # Chilon of Sparta
    "Q267195",    # Cynisca
    "Q705367",    # Menkheperre
    "Q514435",    # Ecclesia (Sparta)
    "Q27697637",  # Democracy in classical Iran
    "Q109432",    # Ephor
    "Q495552",    # Gerousia
    "Q313869",    # Intef II
    "Q211326",    # Lycurgus
    "Q237614",    # Nimrud
    "Q3335071",   # Spartan Constitution
    "Q1418190",   # Spartan army
    "Q369194",    # Spartiate
}

def clean_qid(x):
    m = re.search(r"Q\d+", str(x))
    return m.group(0) if m else ""

def row_qid(row):
    for key in ["qid", "wikidata_qid", "entity", "wikidata_id"]:
        qid = clean_qid(row.get(key, ""))
        if qid:
            return qid
    return ""

if "candidate_rows" not in globals():
    raise NameError("candidate_rows non esiste: devi prima runnare la cella 8.")

if "SOURCE_WIKI_DATASET" not in globals():
    SOURCE_WIKI_DATASET = "DragonLLM/Clean-Wikipedia-English-Articles"

current_qids = {row_qid(r) for r in candidate_rows}
needed_qids = set(FORCE_INCLUDE_QIDS) - current_qids

print("candidate_rows current:", len(candidate_rows))
print("force qids already present:", len(FORCE_INCLUDE_QIDS - needed_qids))
print("force qids still needed:", len(needed_qids))

rescued_rows = []

if needed_qids:
    source = load_dataset(
        SOURCE_WIKI_DATASET,
        split=SOURCE_WIKI_SPLIT,
        streaming=True,
    )

    scanned = 0

    for row in source:
        scanned += 1
        qid = row_qid(row)

        if qid in needed_qids:
            rescued_rows.append(dict(row))
            needed_qids.remove(qid)
            print(f"rescued {len(rescued_rows)}: {qid} | {row.get('title', '')}")

        if not needed_qids:
            break

        if scanned % 250000 == 0:
            print(f"scanned {scanned:,} rows | rescued {len(rescued_rows)} | still needed {len(needed_qids)}")

    print("source rows scanned:", scanned)
    print("rescued rows:", len(rescued_rows))

    if needed_qids:
        print("WARNING: questi QID non sono stati trovati in DragonLLM:")
        for qid in sorted(needed_qids):
            print("-", qid)

candidate_rows.extend(rescued_rows)

# Deduplica finale
df = pd.DataFrame(candidate_rows).fillna("")
df["_qid"] = df.apply(row_qid, axis=1)

if "url" in df.columns:
    df["_dedupe_key"] = df["url"].astype(str).str.lower().str.rstrip("/")
elif "title" in df.columns:
    df["_dedupe_key"] = df["title"].astype(str).str.lower()
else:
    df["_dedupe_key"] = df["_qid"]

df = df.drop_duplicates("_dedupe_key", keep="last")
candidate_rows = df.drop(columns=["_qid", "_dedupe_key"], errors="ignore").to_dict("records")

print("candidate_rows after rescue:", len(candidate_rows))

check_df = pd.DataFrame(candidate_rows).fillna("")
check_df["_qid"] = check_df.apply(row_qid, axis=1)

display(
    check_df[check_df["_qid"].isin(FORCE_INCLUDE_QIDS)]
    [[c for c in ["_qid", "title", "url"] if c in check_df.columns]]
    .sort_values("title")
)

## 9. Align New Rows To Base Schema

New rows keep the base schema. Their taxonomy is assigned from the discovery macro area and marked as pending audit.

In [ ]:
def row_value(row, *keys, default=''):
    for key in keys:
        if isinstance(row, dict) and key in row:
            value = row.get(key)
            if value is not None and not (isinstance(value, str) and value == ''):
                return value
    return default


def token_count(text):
    return len(str(text or '').split())


def feature_default(column):
    feature = base_ds.features[column]
    dtype = getattr(feature, 'dtype', None)
    if getattr(feature, '_type', None) == 'List':
        return []
    if dtype in {'int64', 'int32', 'int16', 'uint64', 'uint32'}:
        return 0
    if dtype in {'float64', 'float32'}:
        return 0.0
    if dtype and str(dtype).startswith('timestamp'):
        return None
    if column == 'categories':
        return []
    return ''

CIV_BY_AREA = {'roman_world': 'Ancient Rome', 'ancient_greece': 'Ancient Greece', 'ancient_egypt': 'Ancient Egypt', 'mesopotamia_near_east': 'Mesopotamia / Near East'}
REGION_BY_AREA = {'roman_world': 'Italy / Rome | Western Mediterranean', 'ancient_greece': 'Greece | Eastern Mediterranean', 'ancient_egypt': 'Egypt', 'mesopotamia_near_east': 'Mesopotamia | Near East'}
ENTITY_TYPE_BY_AREA = {'question_driven': 'concept', 'people': 'person', 'works_texts': 'text_or_work', 'events_warfare': 'event', 'religion_mythology': 'religion_or_deity', 'architecture_places': 'place', 'languages_scripts': 'language', 'law_society_institutions': 'institution', 'technology_science': 'artifact', 'roman_world': 'concept', 'ancient_greece': 'concept', 'ancient_egypt': 'concept', 'mesopotamia_near_east': 'concept'}
TOPIC_BY_AREA = {'question_driven': 'general_history', 'people': 'biography | politics', 'works_texts': 'literature | philosophy | religion', 'events_warfare': 'warfare | politics', 'religion_mythology': 'religion | mythology', 'architecture_places': 'architecture | geography', 'languages_scripts': 'language | literature', 'law_society_institutions': 'law | politics | society', 'technology_science': 'technology | science', 'roman_world': 'general_history', 'ancient_greece': 'general_history', 'ancient_egypt': 'general_history', 'mesopotamia_near_east': 'general_history'}


def infer_taxonomy(meta):
    primary = str(meta.get('primary_macro_area', '') or '').strip()
    areas = split_pipe(meta.get('macro_areas', ''))
    civs, regions = [], []
    for area in areas + [primary]:
        civ = CIV_BY_AREA.get(area)
        region = REGION_BY_AREA.get(area)
        if civ and civ not in civs:
            civs.append(civ)
        if region and region not in regions:
            regions.append(region)
    return {
        'civilization': ' | '.join(civs) if civs else 'Mixed / Broad',
        'entity_type': ENTITY_TYPE_BY_AREA.get(primary, 'concept'),
        'period': 'Mixed / Broad',
        'region': ' | '.join(regions) if regions else 'Mixed / Broad',
        'topic': TOPIC_BY_AREA.get(primary, 'general_history'),
        'taxonomy_confidence': 0.65,
        'taxonomy_source': 'ANCIENT5_wikidata_macro_area_v1',
        'taxonomy_notes': 'auto-assigned from ANCIENT5 QID discovery macro area; audit before hard filters',
    }


def align_to_base_schema(row):
    qid = extract_row_qid(row)
    meta = selected_by_qid.get(qid, {})
    title = str(row_value(row, 'title', default=meta.get('label', '')) or meta.get('label', '')).strip()
    url = canonical_url(row_value(row, 'url', 'canonicalurl', default=meta.get('wikipedia_url', ''))) or canonical_url(meta.get('wikipedia_url', ''))
    text = str(row_value(row, 'text', 'extract', 'page_content', default='') or '')
    description = str(row_value(row, 'description', 'short_description', 'wikidata_description', default='') or '')
    taxonomy = infer_taxonomy(meta)
    aligned = {}
    for col in base_ds.column_names:
        if col == 'text':
            aligned[col] = text
        elif col == 'title':
            aligned[col] = title
        elif col == 'url':
            aligned[col] = url
        elif col == 'categories':
            value = row_value(row, 'categories', default=[])
            if isinstance(value, str):
                value = [value] if value else []
            aligned[col] = list(value or [])
        elif col == 'token_count':
            raw = row_value(row, 'token_count', default=None)
            try:
                aligned[col] = int(raw) if raw not in [None, ''] else token_count(text)
            except Exception:
                aligned[col] = token_count(text)
        elif col == 'id':
            raw_id = row_value(row, 'id', 'pageid', default=0)
            try:
                aligned[col] = int(raw_id)
            except Exception:
                aligned[col] = 0
        elif col == 'qid':
            aligned[col] = qid
        elif col == 'entity':
            aligned[col] = qid
        elif col == 'wikidata_label':
            aligned[col] = meta.get('label', '') or title
        elif col == 'wikidata_description':
            aligned[col] = description
        elif col in taxonomy:
            aligned[col] = taxonomy[col]
        else:
            value = row_value(row, col, default=None)
            aligned[col] = value if value not in (None, '') else feature_default(col)
    return aligned

added_rows, manifest_rows, seen_new_keys = [], [], set()

if not RUN_FETCH_AND_BUILD_DATASET:
    print('Skipping alignment because RUN_FETCH_AND_BUILD_DATASET=False')
else:
    for row in candidate_rows:
        qid = extract_row_qid(row)
        meta = selected_by_qid.get(qid, {})
        title = str(row_value(row, 'title', default=meta.get('label', '')) or meta.get('label', '')).strip()
        url = canonical_url(row_value(row, 'url', 'canonicalurl', default=meta.get('wikipedia_url', ''))) or canonical_url(meta.get('wikipedia_url', ''))
        title_norm = normalize_title(title)
        text = str(row_value(row, 'text', 'extract', 'page_content', default='') or '')
        key = qid or url or title_norm
        if qid and qid in base_qids:
            status = 'already_in_base_qid_skipped'
        elif title_norm and title_norm in base_titles:
            status = 'already_in_base_title_skipped'
        elif url and url in base_urls:
            status = 'already_in_base_url_skipped'
        elif key in seen_new_keys:
            status = 'duplicate_candidate_skipped'
        elif not text:
            status = 'empty_text_skipped'
        else:
            status = 'added'
            seen_new_keys.add(key)
            added_rows.append(align_to_base_schema(row))
        manifest_rows.append({
            'status': status,
            'qid': qid,
            'label': meta.get('label', title),
            'resolved_title': title,
            'wikipedia_url': url,
            'primary_macro_area': meta.get('primary_macro_area', ''),
            'macro_areas': meta.get('macro_areas', ''),
            'source_queries': meta.get('source_queries', ''),
            'source_channels': meta.get('source_channels', ''),
            'sitelinks': meta.get('sitelinks', ''),
            'selection_score': meta.get('selection_score', ''),
            'question_ids': meta.get('question_ids', ''),
            'token_count': token_count(text),
        })
    found_qids = {m['qid'] for m in manifest_rows if m.get('qid')}
    for _, meta in missing_after_fetch.iterrows():
        qid = clean_qid(meta.get('qid', ''))
        if qid not in found_qids:
            manifest_rows.append({
                'status': 'not_found_in_source',
                'qid': qid,
                'label': meta.get('label', ''),
                'resolved_title': '',
                'wikipedia_url': meta.get('wikipedia_url', ''),
                'primary_macro_area': meta.get('primary_macro_area', ''),
                'macro_areas': meta.get('macro_areas', ''),
                'source_queries': meta.get('source_queries', ''),
                'source_channels': meta.get('source_channels', ''),
                'sitelinks': meta.get('sitelinks', ''),
                'selection_score': meta.get('selection_score', ''),
                'question_ids': meta.get('question_ids', ''),
                'token_count': 0,
            })
    added_manifest = pd.DataFrame(manifest_rows)
    display(added_manifest['status'].value_counts(dropna=False).to_frame('rows'))
    display(added_manifest.sort_values(['status', 'primary_macro_area', 'label']).head(100))
    print('added rows ready:', len(added_rows))

## 10. Save ANCIENT5 Dataset

In [ ]:
if not RUN_FETCH_AND_BUILD_DATASET:
    print('Skipping dataset save because RUN_FETCH_AND_BUILD_DATASET=False')
else:
    if OUTPUT_DATASET_DIR.exists() and not OVERWRITE_OUTPUT_DATASET:
        raise FileExistsError(f'{OUTPUT_DATASET_DIR} already exists. Set OVERWRITE_OUTPUT_DATASET=True only if you want to regenerate it.')
    if not added_rows:
        raise RuntimeError('No new rows were added. Check selected QIDs, DragonLLM source access, and base overlap.')

    added_dataset = Dataset.from_list(added_rows, features=base_ds.features)
    combined_dataset = concatenate_datasets([base_ds, added_dataset])

    seen = set()
    def keep_first_occurrence(row):
        qid = clean_qid(row.get('qid', '') or row.get('entity', ''))
        key = ('qid:' + qid) if qid else (canonical_url(row.get('url', '')) or normalize_title(row.get('title', '')))
        if key in seen:
            return False
        seen.add(key)
        return True

    combined_dataset = combined_dataset.filter(keep_first_occurrence)

    if OUTPUT_DATASET_DIR.exists() and OVERWRITE_OUTPUT_DATASET:
        import shutil
        shutil.rmtree(OUTPUT_DATASET_DIR)

    DatasetDict({'train': combined_dataset}).save_to_disk(str(OUTPUT_DATASET_DIR))
    added_manifest.to_csv(ADDED_ARTICLES_CSV, index=False)
    added_manifest[added_manifest['status'] != 'added'].to_csv(SKIPPED_ARTICLES_CSV, index=False)

    report = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'run_label': RUN_LABEL,
        'base_dataset_dir': str(BASE_DATASET_DIR),
        'output_dataset_dir': str(OUTPUT_DATASET_DIR),
        'source_wiki_dataset': SOURCE_WIKI_DATASET,
        'selection_strategy': 'ancient3_v4 seed + Wikidata broad QID discovery + question-driven QIDs; DragonLLM QID filtering',
        'base_rows': base_ds.num_rows,
        'candidate_qids': int(len(candidates)),
        'selected_qids_total': int(len(selected)),
        'selected_new_qids': int(len(selected_new)),
        'candidate_rows_fetched': int(len(candidate_rows)),
        'added_rows': int((added_manifest['status'] == 'added').sum()),
        'not_found_in_source': int((added_manifest['status'] == 'not_found_in_source').sum()),
        'output_rows': combined_dataset.num_rows,
        'candidate_qids_csv': str(CANDIDATE_QIDS_CSV),
        'selected_qids_csv': str(SELECTED_QIDS_CSV),
        'added_articles_csv': str(ADDED_ARTICLES_CSV),
        'skipped_articles_csv': str(SKIPPED_ARTICLES_CSV),
    }
    BUILD_REPORT_JSON.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
    print(json.dumps(report, indent=2, ensure_ascii=False))
    print('saved dataset:', OUTPUT_DATASET_DIR)

## 11. Sanity Check

In [ ]:
if OUTPUT_DATASET_DIR.exists():
    check_ds = load_from_disk(str(OUTPUT_DATASET_DIR))['train']
    check_df = check_ds.to_pandas().fillna('')
    print(check_ds)
    print('rows:', check_ds.num_rows)
    if 'qid' in check_df.columns:
        print('unique qids:', check_df['qid'].map(clean_qid).replace('', pd.NA).dropna().nunique())
    for col in ['civilization', 'entity_type', 'topic']:
        if col in check_df.columns:
            display(check_df[col].value_counts(dropna=False).head(20).to_frame('rows'))
    cols = [c for c in ['title', 'qid', 'civilization', 'entity_type', 'topic', 'url'] if c in check_df.columns]
    display(check_df[cols].tail(30) if cols else check_df.tail(30))
else:
    print('ANCIENT5 dataset does not exist yet. Build is probably still disabled.')

In [ ]:
# @title Install Ollama in this runtime

!apt-get update -qq
!apt-get install -y -qq zstd curl

!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# @title Start Ollama server

import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(8)

try:
    response = requests.get("http://localhost:11434/api/tags", timeout=10)
    print("Ollama status:", response.status_code)
    print(response.json())
except Exception as e:
    print("Ollama non risponde ancora:", repr(e))

In [ ]:
# @title Pull embedding model

!ollama pull hf.co/unsloth/embeddinggemma-300m-GGUF:BF16

In [ ]:
# @title Test embedding endpoint

from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1/",
    api_key="ollama",
)

test = client.embeddings.create(
    model="hf.co/unsloth/embeddinggemma-300m-GGUF:BF16",
    input="Ancient Greece and Sparta"
)

print("embedding dimensions:", len(test.data[0].embedding))
print("first values:", test.data[0].embedding[:5])

## 12. Optional Incremental Index Build

Disabled by default. After ANCIENT5 is saved, this copies `Indexes/ancient_v4` and adds embeddings only for the new ANCIENT5 articles. You need Ollama running in this same runtime.

In [ ]:
# @title Install indexing Python dependencies

!pip install -q \
  langchain-openai \
  langchain-community \
  langchain-text-splitters \
  faiss-cpu \
  tiktoken

In [ ]:
# @title Check indexing imports

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import TokenTextSplitter

print("Indexing imports OK")

In [ ]:
RUN_INCREMENTAL_INDEX = True
OVERWRITE_INDEX = False
SOURCE_INDEX_DIR = INDEXES_DIR / 'ancient_v4'
OUTPUT_INDEX_DIR = INDEXES_DIR / 'ANCIENT5'

if not RUN_INCREMENTAL_INDEX:
    print('Skipping index build because RUN_INCREMENTAL_INDEX=False')
else:
    import shutil
    from tqdm.auto import tqdm
    from langchain_openai import OpenAIEmbeddings
    from langchain_community.vectorstores import FAISS
    from langchain_text_splitters import TokenTextSplitter
    from langchain_core.documents import Document

    if not OUTPUT_DATASET_DIR.exists():
        raise FileNotFoundError(f'Build dataset first: {OUTPUT_DATASET_DIR}')
    if not SOURCE_INDEX_DIR.exists():
        raise FileNotFoundError(f'Source index not found: {SOURCE_INDEX_DIR}')
    if OUTPUT_INDEX_DIR.exists():
        if not OVERWRITE_INDEX:
            raise FileExistsError(f'{OUTPUT_INDEX_DIR} exists. Set OVERWRITE_INDEX=True to replace it.')
        shutil.rmtree(OUTPUT_INDEX_DIR)

    shutil.copytree(SOURCE_INDEX_DIR, OUTPUT_INDEX_DIR)
    manifest = pd.read_csv(ADDED_ARTICLES_CSV).fillna('')
    added_qids = set(manifest.loc[manifest['status'] == 'added', 'qid'].map(clean_qid))
    ds = load_from_disk(str(OUTPUT_DATASET_DIR))['train']
    df = ds.to_pandas().fillna('')
    new_df = df[df['qid'].map(clean_qid).isin(added_qids)].copy()

    embeddings = OpenAIEmbeddings(model='hf.co/unsloth/embeddinggemma-300m-GGUF:BF16', base_url='http://localhost:11434/v1/', api_key='ollama', check_embedding_ctx_length=False)
    vectorstore = FAISS.load_local(str(OUTPUT_INDEX_DIR), embeddings, allow_dangerous_deserialization=True)
    splitter = TokenTextSplitter(encoding_name='cl100k_base', chunk_size=512, chunk_overlap=128)
    metadata_cols = ['id', 'title', 'url', 'categories', 'token_count', 'qid', 'wikidata_label', 'wikidata_description', 'civilization', 'entity_type', 'period', 'region', 'topic', 'taxonomy_confidence', 'taxonomy_source', 'taxonomy_notes']

    docs = []
    for _, row in new_df.iterrows():
        text = str(row.get('text', '') or '')
        meta = {col: row.get(col, '') for col in metadata_cols if col in new_df.columns}
        chunks = splitter.split_text(text)
        for chunk in chunks:
            docs.append(Document(page_content=chunk, metadata=meta))
        print(f"{row.get('title', '')}: {len(chunks)} chunks | qid={row.get('qid', '')}")

    print('new chunks to embed:', len(docs))
    for start in tqdm(range(0, len(docs), 64), desc='embedding ANCIENT5 new chunks'):
        vectorstore.add_documents(docs[start:start + 64])
    vectorstore.save_local(str(OUTPUT_INDEX_DIR))

    report = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'mode': 'incremental_index_from_ancient_v4',
        'source_index_dir': str(SOURCE_INDEX_DIR),
        'output_index_dir': str(OUTPUT_INDEX_DIR),
        'dataset_dir': str(OUTPUT_DATASET_DIR),
        'added_articles_csv': str(ADDED_ARTICLES_CSV),
        'new_article_rows': len(new_df),
        'new_chunks': len(docs),
    }
    index_report = LOGS_DIR / 'ANCIENT5_index_report.json'
    index_report.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
    print(json.dumps(report, indent=2, ensure_ascii=False))